[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jrpmichel/Libro_Regularizacion_La_Salle_Bajio/blob/main/dif/u3_transformaciones.ipynb)

# Dif. U3 · Transformaciones gráficas
**Libro de regularización en matemáticas para ingeniería · ULSA Bajío**

Este notebook acompaña la unidad 3 del bloque de cálculo diferencial. Tiene una sección por subtema, en el mismo orden que el libro:

1. La forma general $y=a\,f\big(b(x-h)\big)+k$
2. Traslaciones y reflexiones
3. Estiramientos y compresiones
4. Composición de transformaciones

**Cómo usarlo.** Ejecuta las celdas en orden (Entorno de ejecución → Ejecutar todas). Cada sección termina con una celda **Contrasta**: antes de ejecutarla, calcula a mano el caso que se indica y escribe tu resultado. Para saber hacia dónde se mueve una gráfica, sigue un punto: es más confiable que la intuición.

**Etiqueta del repositorio.** `DIF-U3` (bloque, unidad). Las celdas de código de cada sección vienen de `dif/u3_transformaciones/NN_*.py`; los fragmentos que el libro imprime, de `dif/u3_transformaciones/fragmentos/`.

**Convenciones.** Punto decimal. Los valores se *redondean* (no se truncan) a las cifras significativas que elijas. Para escribir expresiones usa `x` como variable, `*` para multiplicar, `**` o `^` para potencias, `sqrt(...)` para raíces, `abs(...)` para el valor absoluto, `exp(...)` para $e^x$ y `log(...)` para el logaritmo natural. Los ángulos de `sin`, `cos` y `tan` van en **radianes**.

## 0. Versiones e imports

In [ ]:
# Versiones mínimas probadas. En Colab ya vienen instaladas; esta celda solo lo asegura.
%pip install -q "numpy>=1.26" "sympy>=1.12" "matplotlib>=3.8" "ipywidgets>=8"

import sys, math, re
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import sympy as sp
import ipywidgets as widgets
from IPython.display import display
from sympy.parsing.sympy_parser import (parse_expr, standard_transformations,
                                        implicit_multiplication_application, convert_xor)
from sympy.calculus.util import continuous_domain, function_range

print("Python", sys.version.split()[0], "| numpy", np.__version__, "| sympy", sp.__version__,
      "| matplotlib", matplotlib.__version__, "| ipywidgets", widgets.__version__)

In [ ]:
# Símbolos y utilidades compartidas por todo el notebook
x = sp.symbols("x", real=True)

_LOCAL = {"x": x, "sqrt": sp.sqrt, "abs": sp.Abs, "exp": sp.exp, "log": sp.log,
          "pi": sp.pi, "sin": sp.sin, "cos": sp.cos, "tan": sp.tan, "E": sp.E}
_GLOBAL = {"Integer": sp.Integer, "Float": sp.Float, "Rational": sp.Rational,
           "Symbol": sp.Symbol, "Function": sp.Function}
_TRANSF = standard_transformations + (implicit_multiplication_application, convert_xor)


def parsear(texto):
    """Convierte un texto en expresión de sympy en la variable x, o explica qué salió mal."""
    if not isinstance(texto, str) or not texto.strip():
        raise ValueError("Escribe una expresión, por ejemplo: (x**3 - 1)/(x - 1)")
    desconocidos = sorted(set(re.findall(r"[A-Za-z_]\w*", texto)) - set(_LOCAL))
    if desconocidos:
        raise ValueError(f"No reconozco {desconocidos}. Usa x como variable y, si hace falta, "
                         "sqrt, abs, exp, log (natural), sin, cos, tan o pi.")
    try:
        expr = parse_expr(texto, local_dict=_LOCAL, global_dict=dict(_GLOBAL),
                          transformations=_TRANSF)
    except Exception as err:
        raise ValueError("No pude leer la expresión. Usa x como variable, * para multiplicar, "
                         "** o ^ para potencias y sqrt(...) para raíces.") from err
    if expr.atoms(sp.core.function.AppliedUndef):
        raise ValueError("Hay una función que no conozco. Las permitidas son sqrt, abs, exp, log, sin, cos y tan.")
    libres = {str(s) for s in expr.free_symbols} - {"x"}
    if libres:
        raise ValueError(f"Símbolos no permitidos: {sorted(libres)}. Solo puedes usar x.")
    return expr


def numero(texto):
    """Lee un número escrito como 2, -0.5, 1/3, pi/2 o sqrt(2)."""
    try:
        v = sp.sympify(str(texto).replace("^", "**"), locals={"pi": sp.pi, "sqrt": sp.sqrt, "E": sp.E})
    except (sp.SympifyError, TypeError):
        raise ValueError(f"'{texto}' no es un número.") from None
    if v.free_symbols or not v.is_real:
        raise ValueError(f"'{texto}' no es un número real.")
    return v


def cifras(valor, n=4):
    """Redondea (no trunca) a n cifras significativas y devuelve texto."""
    if valor is None:
        return "no existe"
    if valor in (sp.oo, sp.S.Infinity) or (isinstance(valor, float) and valor == math.inf):
        return "+∞"
    if valor in (-sp.oo, sp.S.NegativeInfinity) or (isinstance(valor, float) and valor == -math.inf):
        return "-∞"
    v = float(valor)
    if math.isnan(v):
        return "no definida"
    if v == 0:
        return "0"
    return np.format_float_positional(v, precision=n, unique=False, fractional=False, trim="-")


def a_texto(valor):
    """Límite de sympy en texto: número exacto si es racional, ±∞ o 'no existe'."""
    if valor is None or valor is sp.nan:
        return "no existe"
    if valor == sp.oo:
        return "+∞"
    if valor == -sp.oo:
        return "-∞"
    if valor == sp.zoo:
        return "∞ sin signo (no existe)"
    if isinstance(valor, sp.AccumBounds):
        return "no existe (oscila)"
    return str(valor) if valor.is_Rational else f"{valor} ≈ {cifras(valor, 6)}"


def conjunto_a_texto(s):
    """Intervalos de sympy en notación de intervalos."""
    if s == sp.S.Reals:
        return "(-∞, ∞) = ℝ"
    if s == sp.S.EmptySet:
        return "∅ (vacío)"
    if isinstance(s, sp.Union):
        return " ∪ ".join(conjunto_a_texto(t) for t in sorted(s.args, key=lambda t: float(t.inf)))
    if isinstance(s, sp.Interval):
        f = lambda p: ("∞" if p == sp.oo else "-∞" if p == -sp.oo
                       else str(p) if p.is_Rational else "≈" + cifras(float(p), 4))
        return ("(" if s.left_open else "[") + f(s.inf) + ", " + f(s.sup) + (")" if s.right_open else "]")
    return str(s)


def evaluar(f, v):
    """f(v) en punto flotante; nan si f no está definida en v."""
    with np.errstate(all="ignore"):
        try:
            r = complex(f(v))
        except (ZeroDivisionError, ValueError, OverflowError):
            return math.nan
    return r.real if abs(r.imag) < 1e-12 else math.nan


def cerca(u, v, tol=1e-9):
    return abs(u - v) < tol


def _rechaza(fn):
    """True si fn() lanza ValueError (para probar que una calculadora rechaza una entrada)."""
    try:
        fn()
    except ValueError:
        return True
    return False


# Funciones base del libro, con tres puntos clave de cada una (x0, y0)
BASES = {
    "x²":    (x**2,        [(-1, 1), (0, 0), (2, 4)]),
    "|x|":   (sp.Abs(x),   [(-1, 1), (0, 0), (2, 2)]),
    "√x":    (sp.sqrt(x),  [(0, 0), (1, 1), (4, 2)]),
    "1/x":   (1/x,         [(-1, -1), (1, 1), (2, sp.Rational(1, 2))]),
    "e^x":   (sp.exp(x),   [(-1, sp.exp(-1)), (0, 1), (1, sp.E)]),
    "sen x": (sp.sin(x),   [(0, 0), (sp.pi/2, 1), (sp.pi, 0)]),
}


def transformar(f, a, b, h, k):
    """g(x) = a f(b(x - h)) + k. Rechaza a = 0 o b = 0, que aplastan la gráfica."""
    if a == 0:
        raise ValueError("a = 0 convierte la gráfica en la recta y = k: ya no es una transformación de f.")
    if b == 0:
        raise ValueError("b = 0 convierte la función en la constante a·f(0) + k (o en nada, si f(0) no existe): "
                         "ya no es una transformación de f.")
    a, b, h, k = (sp.nsimplify(v) for v in (a, b, h, k))
    return a * f.subs(x, b * (x - h)) + k


def imagen(punto, a, b, h, k):
    """(x0, y0) de la gráfica de f  ->  (h + x0/b, k + a·y0) de la gráfica de g."""
    x0, y0 = punto
    return (sp.nsimplify(h) + sp.sympify(x0) / sp.nsimplify(b), sp.nsimplify(k) + sp.nsimplify(a) * sp.sympify(y0))

## 1. La forma general $y=a\,f\big(b(x-h)\big)+k$

Cada punto $(x_0,y_0)$ de la gráfica de $f$ tiene una imagen en la gráfica de $g$:
$$(x_0,\,y_0)\ \longmapsto\ \left(h+\frac{x_0}{b},\ k+a\,y_0\right).$$
Los parámetros de **afuera** ($a$, $k$) actúan sobre $y$ tal como se leen. Los de **adentro** ($b$, $h$) actúan sobre $x$: se divide entre $b$ y se suma $h$.

La calculadora aplica la transformación a una función base del menú, muestra la imagen de tres puntos clave, el dominio y el rango, y dibuja las dos gráficas.

In [ ]:
def calculadora_forma(base, a, b, h, k, n_cifras):
    try:
        f, puntos = BASES[base]
        g = transformar(f, a, b, h, k)
    except (KeyError, ValueError) as err:
        print("Revisa la entrada:", err); return
    print(f"f(x) = {f}      g(x) = {g}")
    print("Puntos clave (redondeados a", n_cifras, "cifras significativas):")
    for p in puntos:
        q = imagen(p, a, b, h, k)
        print(f"   ({cifras(p[0], n_cifras)}, {cifras(p[1], n_cifras)})  ->  ({cifras(q[0], n_cifras)}, {cifras(q[1], n_cifras)})"
              f"   comprobación g = {cifras(g.subs(x, q[0]), n_cifras)}")
    try:
        print("dominio de g:", conjunto_a_texto(continuous_domain(g, x, sp.S.Reals)))
        print("rango de g  :", conjunto_a_texto(function_range(g, x, continuous_domain(g, x, sp.S.Reals))))
    except Exception:
        print("sympy no pudo calcular el dominio o el rango en este caso.")
    fn, gn = sp.lambdify(x, f, "numpy"), sp.lambdify(x, g, "numpy")
    xs = np.linspace(-8, 8, 2001)
    with np.errstate(all="ignore"):
        yf = np.broadcast_to(fn(xs), xs.shape).astype(float)
        yg = np.broadcast_to(gn(xs), xs.shape).astype(float)
    yf = np.where(np.abs(yf) < 20, yf, np.nan); yg = np.where(np.abs(yg) < 20, yg, np.nan)
    plt.figure(figsize=(5, 3.2)); plt.plot(xs, yf, color="gray", label="f (base)")
    plt.plot(xs, yg, "b--", label="g (transformada)")
    plt.axhline(0, color="k", lw=0.6); plt.axvline(0, color="k", lw=0.6)
    plt.ylim(-10, 10); plt.grid(True); plt.legend(fontsize=8); plt.show()


widgets.interact(calculadora_forma,
    base=widgets.Dropdown(options=list(BASES), value="√x", description="base"),
    a=widgets.FloatSlider(value=-2, min=-4, max=4, step=0.5, description="a"),
    b=widgets.FloatSlider(value=2, min=-4, max=4, step=0.5, description="b"),
    h=widgets.FloatSlider(value=1, min=-5, max=5, step=0.5, description="h"),
    k=widgets.FloatSlider(value=3, min=-5, max=5, step=0.5, description="k"),
    n_cifras=widgets.IntSlider(value=4, min=1, max=8, description="cifras sig."));

In [ ]:
# Casos de prueba de la sección 1 (resultado conocido)
def _img(base, a, b, h, k):
    return [imagen(p, a, b, h, k) for p in BASES[base][1]]

g31 = transformar(sp.sqrt(x), -2, 2, 1, 3)
PRUEBAS_1 = [
    ("√x con a=-2, b=2, h=1, k=3: (0,0)->(1,3), (1,1)->(3/2,1), (4,2)->(3,-1)",
     lambda: _img("√x", -2, 2, 1, 3) == [(1, 3), (sp.Rational(3, 2), 1), (3, -1)]),
    ("la imagen de cada punto está en la gráfica de g",
     lambda: all(sp.simplify(g31.subs(x, q[0]) - q[1]) == 0 for q in _img("√x", -2, 2, 1, 3))),
    ("dominio [1, ∞) y rango (-∞, 3]",
     lambda: continuous_domain(g31, x, sp.S.Reals) == sp.Interval(1, sp.oo)
             and function_range(g31, x, sp.Interval(1, sp.oo)) == sp.Interval(-sp.oo, 3)),
    ("x² con h=3: el vértice (0,0) va a (3,0)", lambda: _img("x²", 1, 1, 3, 0)[1] == (3, 0)),
    ("sen x con b=2: (π/2, 1) va a (π/4, 1)", lambda: _img("sen x", 1, 2, 0, 0)[1] == (sp.pi/4, 1)),
    ("1/x con h=3, k=2: dominio sin el 3",
     lambda: continuous_domain(transformar(1/x, 1, 1, 3, 2), x, sp.S.Reals)
             == sp.Union(sp.Interval.open(-sp.oo, 3), sp.Interval.open(3, sp.oo))),
    ("a = 0 se rechaza con mensaje", lambda: _rechaza(lambda: transformar(x**2, 0, 1, 0, 0))),
    ("b = 0 se rechaza con mensaje", lambda: _rechaza(lambda: transformar(x**2, 1, 0, 0, 0))),
]
for nombre, prueba in PRUEBAS_1:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 1).** Para $g(x)=2\,|x-1|+3$ (base $|x|$, con $a=2$, $b=1$, $h=1$, $k=3$), calcula a mano la imagen del punto $(-1,1)$ y comprueba que está en la gráfica de $g$. Escribe tu resultado y compáralo con la calculadora.

In [ ]:
mi_imagen = None       # escribe una pareja (x, y), por ejemplo: (1, 2)

ref = imagen((-1, 1), 2, 1, 1, 3)
print("La calculadora da:", ref, "| g en ese x:", transformar(sp.Abs(x), 2, 1, 1, 3).subs(x, ref[0]))
print("falta tu cálculo" if mi_imagen is None else
      ("coincide" if all(cerca(float(u), float(v)) for u, v in zip(mi_imagen, ref)) else "NO coincide: ¿dividiste entre b y sumaste h?"))

## 2. Traslaciones y reflexiones

- $f(x-h)$ mueve la gráfica $h$ unidades a la **derecha** si $h>0$ (a la izquierda si $h<0$).
- $f(x)+k$ la mueve $k$ unidades hacia **arriba** si $k>0$.
- $-f(x)$ la refleja respecto al **eje $x$**; $f(-x)$, respecto al **eje $y$**.

La primera calculadora describe con palabras lo que hace cada cambio y lo comprueba con un punto. La segunda decide si una función es par, impar o ninguna de las dos comparando $f(-x)$ con $f(x)$.

In [ ]:
def describir(h, k, refleja_x, refleja_y):
    """Texto con la secuencia de movimientos de y = s_x·f(s_y·(x - h)) + k."""
    pasos = []
    if refleja_y: pasos.append("reflejar respecto al eje y (cambiar x por -x)")
    if h: pasos.append(f"mover {abs(h)} a la {'derecha' if h > 0 else 'izquierda'}")
    if refleja_x: pasos.append("reflejar respecto al eje x (cambiar y por -y)")
    if k: pasos.append(f"mover {abs(k)} hacia {'arriba' if k > 0 else 'abajo'}")
    return pasos or ["ningún cambio"]


def calculadora_movimientos(base, h, k, refleja_x, refleja_y):
    f, puntos = BASES[base]
    a, b = (-1 if refleja_x else 1), (-1 if refleja_y else 1)
    g = transformar(f, a, b, h, k)
    print("g(x) =", g)
    for i, p in enumerate(describir(h, k, refleja_x, refleja_y), 1):
        print(f"  {i}. {p}")
    p0 = puntos[1]; q = imagen(p0, a, b, h, k)
    print(f"Comprobación con un punto: {p0} de f  ->  {q} de g;  g({q[0]}) = {sp.simplify(g.subs(x, q[0]))}")


widgets.interact(calculadora_movimientos,
    base=widgets.Dropdown(options=list(BASES), value="x²", description="base"),
    h=widgets.IntSlider(value=3, min=-5, max=5, description="h"),
    k=widgets.IntSlider(value=-1, min=-5, max=5, description="k"),
    refleja_x=widgets.Checkbox(value=False, description="reflejar en eje x"),
    refleja_y=widgets.Checkbox(value=False, description="reflejar en eje y"));

In [ ]:
def paridad(texto):
    """'par', 'impar' o 'ninguna', comparando f(-x) con f(x) de forma simbólica."""
    f = parsear(texto)
    fm = f.subs(x, -x)
    if sp.simplify(fm - f) == 0:
        return "par"
    if sp.simplify(fm + f) == 0:
        return "impar"
    return "ninguna"


def calculadora_paridad(texto):
    try:
        tipo = paridad(texto)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    print(f"f(x) = {texto}:  {tipo}")
    print("par: simétrica respecto al eje y;  impar: simétrica respecto al origen.")


widgets.interact(calculadora_paridad, texto=widgets.Text(value="x*cos(x)", description="f(x) ="));

In [ ]:
# Casos de prueba de la sección 2 (resultado conocido)
PRUEBAS_2 = [
    ("x³ es impar, cos x es par, x² + x no es ninguna",
     lambda: (paridad("x**3"), paridad("cos(x)"), paridad("x**2 + x")) == ("impar", "par", "ninguna")),
    ("x cos x es impar y e^(-x²) es par", lambda: (paridad("x*cos(x)"), paridad("exp(-x**2)")) == ("impar", "par")),
    ("(x+2)² - 1 tiene vértice (-2, -1)", lambda: imagen((0, 0), 1, 1, -2, -1) == (-2, -1)),
    ("√(-x) tiene dominio (-∞, 0]", lambda: continuous_domain(sp.sqrt(-x), x, sp.S.Reals) == sp.Interval(-sp.oo, 0)),
    ("e^(-x) en x = 1 vale lo que e^x en x = -1", lambda: transformar(sp.exp(x), 1, -1, 0, 0).subs(x, 1) == sp.exp(-1)),
    ("h = 3 se describe como 'mover 3 a la derecha'", lambda: describir(3, 0, False, False) == ["mover 3 a la derecha"]),
    ("una expresión con otra letra se rechaza", lambda: _rechaza(lambda: paridad("t**2"))),
]
for nombre, prueba in PRUEBAS_2:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 2).** Escribe a mano el vértice de $y=(x+3)^2+1$ y di hacia dónde se movió la parábola $y=x^2$. Compara con la calculadora (base $x^2$, $h=-3$, $k=1$).

In [ ]:
mi_vertice = None       # escribe una pareja, por ejemplo: (0, 0)

ref = imagen((0, 0), 1, 1, -3, 1)
print("La calculadora da:", ref)
print("falta tu cálculo" if mi_vertice is None else
      ("coincide" if tuple(mi_vertice) == ref else "NO coincide: ¿qué valor de x hace cero a x + 3?"))

## 3. Estiramientos y compresiones

- $a\,f(x)$ multiplica las alturas por $a$: estira si $|a|>1$, comprime si $0<|a|<1$.
- $f(bx)$ divide las abscisas entre $b$: **comprime** si $|b|>1$ y estira si $0<|b|<1$.

Para una función periódica de periodo $T$, $f(bx)$ tiene periodo $T/|b|$. La calculadora describe $y=a\,\mathrm{sen}\big(b(x-h)\big)+k$: amplitud, periodo, rango y primer máximo a partir de $x=h$.

In [ ]:
def senoide(a, b, h=0, k=0):
    """Datos de y = a sen(b(x - h)) + k, en forma exacta con sympy."""
    a, b, h, k = (sp.nsimplify(v) for v in (a, b, h, k))
    if a == 0 or b == 0:
        raise ValueError("a y b deben ser distintos de cero (con a = 0 o b = 0 la función es constante).")
    T = 2 * sp.pi / abs(b)
    # primer máximo con x >= h: a·sen(u) es máximo en u = π/2 (a > 0) o en u = 3π/2 (a < 0),
    # con u = b(x - h); si b < 0 se toma el mismo ángulo un periodo antes para que x >= h
    u_max = sp.pi / 2 if a > 0 else 3 * sp.pi / 2
    xmax = h + (u_max if b > 0 else u_max - 2 * sp.pi) / b
    return {"amplitud": abs(a), "periodo": T, "rango": (k - abs(a), k + abs(a)), "primer máximo": sp.simplify(xmax)}


def calculadora_senoide(a, b, h, k, n_cifras):
    try:
        d = senoide(a, b, h, k)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    print(f"y = {cifras(a, 4)}·sen({cifras(b, 4)}(x - {cifras(h, 4)})) + {cifras(k, 4)}")
    print(f"amplitud = {cifras(d['amplitud'], n_cifras)}   periodo = {d['periodo']} ≈ {cifras(d['periodo'], n_cifras)}")
    print(f"rango = [{cifras(d['rango'][0], n_cifras)}, {cifras(d['rango'][1], n_cifras)}]   "
          f"primer máximo en x = {d['primer máximo']} ≈ {cifras(d['primer máximo'], n_cifras)}")
    xs = np.linspace(h, h + 2 * float(d["periodo"]), 600)
    plt.figure(figsize=(5, 3)); plt.plot(xs, a * np.sin(b * (xs - h)) + k, "b-")
    plt.axhline(k, color="gray", ls="--"); plt.grid(True); plt.xlabel("x (rad)"); plt.show()


widgets.interact(calculadora_senoide,
    a=widgets.FloatText(value=2.0, description="a"), b=widgets.FloatText(value=3.0, description="b"),
    h=widgets.FloatText(value=0.0, description="h"), k=widgets.FloatText(value=1.0, description="k"),
    n_cifras=widgets.IntSlider(value=5, min=1, max=8, description="cifras sig."));

In [ ]:
# Casos de prueba de la sección 3 (resultado conocido)
PRUEBAS_3 = [
    ("sen 2x tiene periodo π", lambda: senoide(1, 2)["periodo"] == sp.pi),
    ("2 sen 3x + 1: amplitud 2, periodo 2π/3, rango [-1, 3], primer máximo π/6",
     lambda: (lambda d: d["amplitud"] == 2 and d["periodo"] == 2*sp.pi/3 and d["rango"] == (-1, 3)
              and d["primer máximo"] == sp.pi/6)(senoide(2, 3, 0, 1))),
    ("sen(x/2) tiene periodo 4π", lambda: senoide(1, sp.Rational(1, 2))["periodo"] == 4*sp.pi),
    ("-3 sen x: amplitud 3, primer máximo en 3π/2", lambda: senoide(-3, 1)["primer máximo"] == 3*sp.pi/2),
    ("√(x/4) = √x/2 (estirar horizontalmente por 4 = comprimir verticalmente por 1/2)",
     lambda: sp.simplify(sp.sqrt(x/4) - sp.sqrt(x)/2) == 0),
    ("b = 0 se rechaza con mensaje", lambda: _rechaza(lambda: senoide(1, 0))),
]
for nombre, prueba in PRUEBAS_3:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 3).** Calcula a mano el periodo de $y=\mathrm{sen}(x/3)$ y di si la gráfica se comprimió o se estiró respecto a $\mathrm{sen}\,x$. Compara con la calculadora ($a=1$, $b=1/3$).

In [ ]:
mi_periodo = None       # escribe un número, por ejemplo: 3.5 o 2*math.pi

ref = senoide(1, sp.Rational(1, 3))["periodo"]
print("La calculadora da:", ref, "≈", cifras(ref, 6))
print("falta tu cálculo" if mi_periodo is None else
      ("coincide" if cerca(mi_periodo, float(ref), 1e-6) else "NO coincide: el periodo de f(bx) es T/|b|"))

## 4. Composición de transformaciones

Si la fórmula trae $f(bx+c)$, primero se **factoriza** $b$: $f(bx+c)=f\big(b(x-h)\big)$ con $h=-c/b$. El desplazamiento es $h$, no $c$.

La calculadora recibe $b$ y $c$, da $h$ y compara la función correcta con la que resulta de desplazar $c$ en lugar de $h$. También muestra los dos órdenes posibles de los pasos horizontales.

In [ ]:
def factorizar_interior(b, c):
    """h tal que b x + c = b (x - h)."""
    b, c = sp.nsimplify(b), sp.nsimplify(c)
    if b == 0:
        raise ValueError("b = 0: el interior no depende de x.")
    return -c / b


def calculadora_orden(base, b, c):
    try:
        f = BASES[base][0]
        h = factorizar_interior(b, c)
    except (KeyError, ValueError) as err:
        print("Revisa la entrada:", err); return
    b_, c_ = sp.nsimplify(b), sp.nsimplify(c)
    lado = lambda d: f"{abs(d)} a la {'derecha' if d > 0 else 'izquierda'}"
    print(f"f({b_*x + c_}) = f({b_}·(x - ({h})))   ->   h = {h} ≈ {cifras(h, 5)}")
    print(f"Orden A: escalar x por 1/{b_} y después mover {lado(h)}.")
    print(f"Orden B: mover {lado(-c_)} y después escalar x por 1/{b_}.")
    bien = f.subs(x, b_ * x + c_)
    mal = f.subs(x, b_ * (x + c_))
    print("¿mover c en lugar de h da lo mismo?", "sí" if sp.simplify(bien - mal) == 0 else "no")


widgets.interact(calculadora_orden,
    base=widgets.Dropdown(options=list(BASES), value="x²", description="base"),
    b=widgets.FloatText(value=2.0, description="b"), c=widgets.FloatText(value=-6.0, description="c"));

In [ ]:
# Casos de prueba de la sección 4 (resultado conocido)
PRUEBAS_4 = [
    ("(2x - 6): h = 3", lambda: factorizar_interior(2, -6) == 3),
    ("(4x + 8): h = -2", lambda: factorizar_interior(4, 8) == -2),
    ("sen(2x + π/3): h = -π/6", lambda: factorizar_interior(2, sp.pi/3) == -sp.pi/6),
    ("10 sen(120πt - π/3): h = 1/360 s", lambda: factorizar_interior(120*sp.pi, -sp.pi/3) == sp.Rational(1, 360)),
    ("(2x - 6)² y (2(x - 6))² son distintas", lambda: sp.simplify((2*x - 6)**2 - (2*(x - 6))**2) != 0),
    ("2 sen x + 3 y 2(sen x + 3) tienen rangos [1,5] y [4,8]",
     lambda: function_range(2*sp.sin(x) + 3, x, sp.S.Reals) == sp.Interval(1, 5)
             and function_range(2*(sp.sin(x) + 3), x, sp.S.Reals) == sp.Interval(4, 8)),
    ("orden B: mover π/3 y luego comprimir da sen(2x + π/3)",
     lambda: sp.simplify(sp.sin(x + sp.pi/3).subs(x, 2*x) - sp.sin(2*x + sp.pi/3)) == 0),
    ("b = 0 se rechaza", lambda: _rechaza(lambda: factorizar_interior(0, 1))),
]
for nombre, prueba in PRUEBAS_4:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 4).** Escribe a mano $(3x+12)^2$ en la forma $f\big(b(x-h)\big)$ con $f(x)=x^2$. Da $b$ y $h$ y compáralos con la calculadora ($b=3$, $c=12$).

In [ ]:
mi_b, mi_h = None, None      # escribe dos números, por ejemplo: 1, 0

ref = factorizar_interior(3, 12)
print("La calculadora da: b = 3, h =", ref)
print("falta tu cálculo" if mi_h is None else ("coincide" if cerca(mi_h, float(ref)) and mi_b == 3 else "NO coincide: h = -c/b"))

## Verificación final
Ejecuta esta celda al terminar. Debe imprimir que todas las pruebas pasaron.

In [ ]:
todas = PRUEBAS_1 + PRUEBAS_2 + PRUEBAS_3 + PRUEBAS_4
fallas = [nombre for nombre, prueba in todas if not prueba()]
assert not fallas, f"Fallan {len(fallas)} pruebas: {fallas}"
print(f"Todas las verificaciones pasaron: {len(todas)}/{len(todas)}")